In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [2]:
from rag_helper import RAGBase
from ingest import build_index,load_faq_data


In [3]:
documents = load_faq_data()
index=build_index(documents)

In [4]:
def rag(self, query):
    search_results = self.search(query)
    prompt = self.build_prompt(query, search_results)
    answer = self.llm(prompt)
    return answer

In [5]:
# asking without tool
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'Absolutely — in most cases, yes. If the course is still open for enrollment, you can usually join even if you just discovered it.\n\nTo check quickly, look for:\n- an **Enroll / Join / Register** button\n- the **start date** and whether it’s **self-paced** or **live**\n- any **deadline** or **capacity limit**\n\nIf you want, send me the course name or link and I can help you figure out whether you can still join.'

In [6]:
# Defining the tool for the agent to use
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [7]:
# search_tool is a dictionary that defines the structure and parameters of the search function,
# which can be used by an agent to perform searches in the FAQ database. 
# It specifies the type of tool, its name, description, and the expected parameters for the search query.

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [8]:
# sending the question with search tool to the model
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join late enrollment access new student"}', call_id='call_GmSOK31wKD21hrtNeoIpnBHZ', name='search', type='function_call', id='fc_069464551f91f30f006a4a8585a6f8819cbd05f03873adf9bb', namespace=None, status='completed')]

In [9]:
# Executing the function and sending the result back
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [10]:
# sending the result back to the model
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [11]:
# Asking the model again
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

'Yes — you can still join.\n\nIf you want a certificate, make sure to submit your project while submissions are still open.'

In [12]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(866, 29)

In [13]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(770, 34)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0001359


In [14]:
# Agentic loop
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [15]:
# a function call helper
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [16]:
# processing the question with the agentic loop

question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join course enrollment discovered the course can I join"}
function_call: search {"query":"course registration late enrollment can I join discovered course"}


In [17]:
# The function call loop continues until there are no more function calls to process.
 
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
ASSISTANT:
Yes — you can still join. The course materials are available, and you can start learning and submitting homework while the submission form is open.

One important note: if you want a certificate, you need to submit your project while the course is still accepting submissions.

If you’d like, I can also help you figure out the best way to start the course now. Are there other areas you want to explore?


In [18]:
# wrapping the agentic loop into a function for reusability

def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [19]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Olama locally run install local run Ollama course FAQ"}
function_call: search {"query":"run Ollama locally setup install Mac Windows Linux FAQ"}
function_call: search {"query":"Ollama local server start pull model run command FAQ"}
iteration #2...
ASSISTANT:
To run Ollama locally:

1. **Install Ollama**
   - Go to https://ollama.com/download and choose your OS.
   - **macOS:** install the `.pkg`
   - **Windows:** install the `.msi`
   - **Linux:** run:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. **Start a model**
   In a terminal, run:
   ```bash
   ollama run llama3
   ```
   This will download the model if needed and start a local chat session.

3. **Check the local server**
   You can test whether Ollama is running with:
   ```bash
   curl http://localhost:11434
   ```

4. **Use it from Python**
   Install the client:
   ```bash
   pip install ollama
   ```
   Then:
   ```python
   import ollama

   res

'To run Ollama locally:\n\n1. **Install Ollama**\n   - Go to https://ollama.com/download and choose your OS.\n   - **macOS:** install the `.pkg`\n   - **Windows:** install the `.msi`\n   - **Linux:** run:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. **Start a model**\n   In a terminal, run:\n   ```bash\n   ollama run llama3\n   ```\n   This will download the model if needed and start a local chat session.\n\n3. **Check the local server**\n   You can test whether Ollama is running with:\n   ```bash\n   curl http://localhost:11434\n   ```\n\n4. **Use it from Python**\n   Install the client:\n   ```bash\n   pip install ollama\n   ```\n   Then:\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you get a connection error, restarting the server often helps:\n```bash\nnohup ollama serve > nohup.o

In [20]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query":"join course enroll discovered the course can I join late registration FAQ"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

If your goal is to get a certificate, make sure you submit your project while submissions are still open. Otherwise, you can still start learning at any time.

If you want, I can also help with how to start the course or explain the certificate requirements.


'Yes — you can still join the course.\n\nIf your goal is to get a certificate, make sure you submit your project while submissions are still open. Otherwise, you can still start learning at any time.\n\nIf you want, I can also help with how to start the course or explain the certificate requirements.'

In [21]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...


function_call: search {"query":"queen gambit chess opening Queen's Gambit what is it"}
iteration #2...
function_call: search {"query":"Queen's Gambit chess opening accepted declined overview"}
iteration #3...
ASSISTANT:
The **Queen’s Gambit** is a chess opening that starts with:

1. **d4 d5**
2. **c4**

White offers the c-pawn as a “gambit” to try to gain better control of the center, especially the **e5** square.

A common idea is:
- **White** gives up the c-pawn temporarily
- **Black** can accept or decline it
- White aims for strong central play and active pieces

Two main branches:
- **Queen’s Gambit Accepted**: Black takes the c-pawn
- **Queen’s Gambit Declined**: Black does not take it

If you want, I can also explain:
- the difference between Accepted vs Declined,
- the main ideas for White and Black,
- or show a simple example game.


'The **Queen’s Gambit** is a chess opening that starts with:\n\n1. **d4 d5**\n2. **c4**\n\nWhite offers the c-pawn as a “gambit” to try to gain better control of the center, especially the **e5** square.\n\nA common idea is:\n- **White** gives up the c-pawn temporarily\n- **Black** can accept or decline it\n- White aims for strong central play and active pieces\n\nTwo main branches:\n- **Queen’s Gambit Accepted**: Black takes the c-pawn\n- **Queen’s Gambit Declined**: Black does not take it\n\nIf you want, I can also explain:\n- the difference between Accepted vs Declined,\n- the main ideas for White and Black,\n- or show a simple example game.'

In [22]:
# instructions updated to dont answer off-topic questions

instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit chess opening queen gambit"}
iteration #2...
function_call: search {"query":"queen gambit"}
iteration #3...
ASSISTANT:
I couldn’t find anything in the course FAQ about “queen gambit,” so this looks off-topic for this course.

If you meant something else course-related, tell me and I’ll look it up. Are there other areas you want to explore?


'I couldn’t find anything in the course FAQ about “queen gambit,” so this looks off-topic for this course.\n\nIf you meant something else course-related, tell me and I’ll look it up. Are there other areas you want to explore?'

In [23]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [24]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [25]:
# letting the agent to use the search tool to answer questions about the course

def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [26]:
# register without passing schema

agent_tools = Tools()
agent_tools.add_tool(search)

In [27]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [28]:
# chat interface and runner setup for the agentic loop

chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [29]:
# running the agentic loop with the runner and displaying the conversation in the chat interface

result = runner.loop(
    prompt="How do I run Ollama locally?",
    callback=callback,
)

-> Response received


-> Response received


-> Response received


In [30]:
# trying with typing error

result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


-> Response received


In [31]:
# cost
result.cost

CostInfo(input_cost=Decimal('0.00300975'), output_cost=Decimal('0.001395'), total_cost=Decimal('0.00440475'))

In [32]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"Olama run locally local setup install FAQ"}', call_id='call_BE

In [33]:
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


-> Response received


In [34]:
runner.run()

-> Response received


-> Response received


Chat ended.


LoopResult(new_messages=[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None), EasyInputMessage(content='', role='user', phase=None, type=None), ResponseFunctionToolCall(arguments='{"query":"course logistics FAQ"}', call_id='call_Rp568bAmBhDQbTniNIIAf6dR', n